In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Project imports
PROJECT_ROOT = Path(r"c:\Users\nikhi\Desktop\CXR_LT_ISBI")
sys.path.insert(0, str(PROJECT_ROOT))

import config
from dataloader import create_dataloaders, CXRDataset, get_val_transforms
from models import get_model, DenseNet121Classifier, ResNet50Classifier
from trainer import predict
from metrics import compute_metrics
from utils import set_seed, create_submission

# Phase 4 modules
from tta import TTAPredictor, MultiModelTTA, TTA_CONFIGS, tta_predict
from threshold_optimizer import (
    ThresholdOptimizer, find_optimal_thresholds_per_class,
    evaluate_threshold_strategies, apply_thresholds
)
from calibration import (
    Calibrator, TemperatureScaling, compare_calibration_methods,
    compute_ece, compute_mce
)

set_seed(config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load Trained Model and Data

In [ ]:
# Load training data for validation split
train_df = pd.read_csv(config.TRAIN_CSV)
print(f"Total samples: {len(train_df)}")

# Create dataloaders (we need validation set for threshold/calibration tuning)
train_loader, val_loader = create_dataloaders(
    train_df,
    config.IMAGE_DIR,
    config.CLASS_NAMES,
    batch_size=config.BATCH_SIZE,
    image_size=config.IMAGE_SIZE,
    val_split=0.1,
    num_workers=config.NUM_WORKERS
)

print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Load your best trained model
# UPDATE THIS PATH to your best checkpoint
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "best_model.pth"

# Choose the matching model architecture
MODEL_NAME = "densenet121"  # or "resnet50", "efficientnet_b4", etc.

model = get_model(MODEL_NAME, num_classes=len(config.CLASS_NAMES), pretrained=False)

if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded checkpoint from {CHECKPOINT_PATH}")
else:
    print(f"WARNING: Checkpoint not found at {CHECKPOINT_PATH}")
    print("Using random weights for demonstration")

model = model.to(device)
model.eval()
print(f"Model loaded: {MODEL_NAME}")

## 2. Get Baseline Predictions

In [ ]:
# Get validation predictions (without TTA)
print("Getting baseline predictions...")

val_logits = []
val_labels = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        labels = batch['labels'].numpy()
        
        logits = model(images)
        val_logits.append(logits.cpu().numpy())
        val_labels.append(labels)

val_logits = np.concatenate(val_logits, axis=0)
val_labels = np.concatenate(val_labels, axis=0)
val_probs = 1 / (1 + np.exp(-val_logits))  # sigmoid

print(f"Validation set: {len(val_logits)} samples")

# Baseline metrics
baseline_metrics = compute_metrics(val_probs, val_labels)
print(f"\nBaseline (no TTA, no calibration):")
print(f"  mAP: {baseline_metrics['mAP']:.4f}")
print(f"  mAUC: {baseline_metrics['mAUC']:.4f}")
print(f"  ECE: {compute_ece(val_probs, val_labels):.4f}")

---
## 3. Test-Time Augmentation (TTA)

TTA applies multiple augmentations and averages predictions.

In [ ]:
# Show available TTA configurations
print("Available TTA configurations:")
for name, transforms in TTA_CONFIGS.items():
    print(f"  {name}: {len(transforms)} transforms")

In [ ]:
# Compare TTA configurations
print("\nComparing TTA configurations on validation set...")
print("="*60)

tta_results = {}

for tta_name in ["none", "flip", "light", "medium"]:
    print(f"\nTesting TTA: {tta_name}")
    
    tta_probs = tta_predict(
        model=model,
        dataloader=val_loader,
        device=device,
        tta_config=tta_name,
        merge_mode="mean"
    )
    
    metrics = compute_metrics(tta_probs, val_labels)
    tta_results[tta_name] = {
        "mAP": metrics['mAP'],
        "mAUC": metrics['mAUC'],
        "probs": tta_probs
    }
    print(f"  mAP: {metrics['mAP']:.4f}, mAUC: {metrics['mAUC']:.4f}")

# Summary
print("\n" + "="*60)
print("TTA Summary:")
print("-"*40)
for name, result in sorted(tta_results.items(), key=lambda x: x[1]['mAP'], reverse=True):
    improvement = result['mAP'] - baseline_metrics['mAP']
    print(f"  {name:15s}: mAP={result['mAP']:.4f} ({improvement:+.4f})")

In [ ]:
# Select best TTA configuration
BEST_TTA = max(tta_results.items(), key=lambda x: x[1]['mAP'])[0]
print(f"Best TTA configuration: {BEST_TTA}")

# Use TTA predictions for subsequent steps
tta_probs = tta_results[BEST_TTA]['probs']

---
## 4. Probability Calibration

Ensure predicted probabilities are well-calibrated.

In [ ]:
# Compare calibration methods
print("Comparing calibration methods...")

calibration_results = compare_calibration_methods(
    logits=val_logits,
    predictions=tta_probs,
    labels=val_labels,
    num_classes=len(config.CLASS_NAMES)
)

In [ ]:
# Check if calibration affects mAP
print("\nEffect of calibration on mAP:")
print("-"*50)

for method, result in calibration_results.items():
    if 'probs' in result:
        metrics = compute_metrics(result['probs'], val_labels)
        print(f"  {method:20s}: mAP={metrics['mAP']:.4f}, ECE={result['ECE']:.4f}")

In [ ]:
# Fit and save calibrator
# Choose method based on ECE reduction and mAP maintenance

CALIBRATION_METHOD = "temperature"  # Usually best for neural networks

calibrator = Calibrator(num_classes=len(config.CLASS_NAMES), method=CALIBRATION_METHOD)
calibrator.fit(val_logits, val_labels)

# Get calibrated predictions
calibrated_probs = calibrator.calibrate(val_logits)

# Evaluate
cal_metrics = compute_metrics(calibrated_probs, val_labels)
print(f"\nCalibrated predictions:")
print(f"  mAP: {cal_metrics['mAP']:.4f}")
print(f"  ECE: {compute_ece(calibrated_probs, val_labels):.4f}")

---
## 5. Threshold Optimization

Find optimal per-class thresholds for binary predictions.

In [ ]:
# Compare threshold strategies
print("Evaluating threshold strategies...")

class_counts = train_df[config.CLASS_NAMES].sum().values

threshold_results = evaluate_threshold_strategies(
    y_true=val_labels,
    y_pred=tta_probs,  # Using TTA predictions
    class_counts=class_counts
)

In [ ]:
# Get per-class optimal thresholds
print("\nOptimal thresholds per class:")

optimizer = ThresholdOptimizer(
    num_classes=len(config.CLASS_NAMES),
    class_names=config.CLASS_NAMES,
    strategy="per_class_f1"
)

optimizer.fit(val_labels, tta_probs, verbose=True)

In [ ]:
# Visualize threshold distribution
thresholds = optimizer.thresholds

plt.figure(figsize=(12, 4))

# Sort by threshold value
sorted_idx = np.argsort(thresholds)

plt.subplot(1, 2, 1)
plt.bar(range(len(thresholds)), thresholds[sorted_idx])
plt.axhline(y=0.5, color='r', linestyle='--', label='Default (0.5)')
plt.xlabel('Class (sorted by threshold)')
plt.ylabel('Optimal Threshold')
plt.title('Optimal Thresholds per Class')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(thresholds, bins=20, edgecolor='black')
plt.axvline(x=0.5, color='r', linestyle='--', label='Default (0.5)')
plt.xlabel('Threshold')
plt.ylabel('Count')
plt.title('Threshold Distribution')
plt.legend()

plt.tight_layout()
plt.show()

print(f"\nThreshold statistics:")
print(f"  Min: {thresholds.min():.3f}")
print(f"  Max: {thresholds.max():.3f}")
print(f"  Mean: {thresholds.mean():.3f}")
print(f"  Median: {np.median(thresholds):.3f}")

---
## 6. Final Test Predictions

Apply TTA + Calibration + Thresholds to test set.

In [ ]:
# Load test data
test_df = pd.read_csv(config.TEST_CSV)
print(f"Test samples: {len(test_df)}")

from torch.utils.data import DataLoader

test_dataset = CXRDataset(
    df=test_df,
    image_dir=config.IMAGE_DIR,
    class_names=config.CLASS_NAMES,
    transform=get_val_transforms(config.IMAGE_SIZE),
    is_test=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

In [ ]:
# Step 1: Get test predictions with TTA
print(f"Running TTA ({BEST_TTA}) on test set...")

test_probs_tta = tta_predict(
    model=model,
    dataloader=test_loader,
    device=device,
    tta_config=BEST_TTA,
    merge_mode="mean"
)

print(f"Test predictions shape: {test_probs_tta.shape}")

In [ ]:
# Step 2: Apply calibration (optional - check if it helps on validation)
# Note: For test set, we need logits for temperature scaling

# If using TTA probabilities directly:
test_probs_final = test_probs_tta

# If you want to apply calibration, get test logits first:
# test_logits = []
# with torch.no_grad():
#     for batch in test_loader:
#         images = batch['image'].to(device)
#         logits = model(images)
#         test_logits.append(logits.cpu().numpy())
# test_logits = np.concatenate(test_logits)
# test_probs_final = calibrator.calibrate(test_logits)

In [ ]:
# Create submission with probabilities (for mAP evaluation)
submission = create_submission(
    test_df=test_df,
    predictions=test_probs_final,
    class_names=config.CLASS_NAMES
)

# Save submission
submission_path = PROJECT_ROOT / "submissions" / "phase4_tta_submission.csv"
submission_path.parent.mkdir(exist_ok=True)
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"\nSubmission preview:")
submission.head()

In [ ]:
# If binary predictions are needed, apply optimized thresholds
binary_predictions = optimizer.predict(test_probs_final)

print(f"Binary predictions per class:")
for i, name in enumerate(config.CLASS_NAMES):
    count = binary_predictions[:, i].sum()
    pct = count / len(binary_predictions) * 100
    print(f"  {name:30s}: {count:4d} ({pct:5.2f}%)")

---
## 7. Multi-Model Ensemble with TTA

In [ ]:
# Example: Ensemble multiple models with TTA
# Uncomment and modify paths to use

# from tta import MultiModelTTA
# 
# # Load multiple trained models
# models = []
# model_configs = [
#     ("densenet121", "checkpoints/densenet121_best.pth"),
#     ("efficientnet_b4", "checkpoints/efficientnet_b4_best.pth"),
#     ("convnext_tiny", "checkpoints/convnext_tiny_best.pth"),
# ]
# 
# for model_name, ckpt_path in model_configs:
#     m = get_model(model_name, num_classes=len(config.CLASS_NAMES), pretrained=False)
#     m.load_state_dict(torch.load(PROJECT_ROOT / ckpt_path)['model_state_dict'])
#     m.to(device)
#     m.eval()
#     models.append(m)
# 
# # Create ensemble with TTA
# ensemble_tta = MultiModelTTA(
#     models=models,
#     model_weights=[0.4, 0.3, 0.3],  # Weighted by validation mAP
#     tta_config="flip",
#     device=device
# )
# 
# # Get ensemble predictions
# ensemble_probs = ensemble_tta.predict(test_loader)
# 
# # Create submission
# ensemble_submission = create_submission(test_df, ensemble_probs, config.CLASS_NAMES)
# ensemble_submission.to_csv(PROJECT_ROOT / "submissions/ensemble_tta_submission.csv", index=False)

---
## Summary

Phase 4 applied:

1. **TTA**: Multiple augmentation configurations compared
2. **Calibration**: Temperature scaling to improve probability calibration
3. **Threshold Optimization**: Per-class thresholds for binary predictions

### Recommended Settings:
- **TTA**: Use "flip" or "light" (good accuracy-speed tradeoff)
- **Calibration**: Temperature scaling (simple, effective)
- **Thresholds**: Per-class F1-optimal (for binary predictions)

### Expected Improvement:
- TTA: +0.5-1.5% mAP
- Calibration: Mainly improves ECE, minor mAP impact
- Thresholds: Improves F1, no impact on mAP (ranking-based)